# Ordinary Differential Equations (ODEs) Pt III
Last week, we covered solving [1st order ODEs](./6a_ODEs.ipynb) of a single independent variable of the form:
$$ \frac{dx}{dt} = f(x,t),$$

And, extended this method to solving [2nd order ODEs](./6b_ODEs.ipynb) 
$$\frac{d^{2}x}{dt^{2}} = f\left(x, \frac{dx}{dt}, t\right)$$
by reducing to two coupled equations of the form:
$$\frac{dy}{dt} = f(x,y,t) ; \ \  \frac{dx}{dt} = y(x,t)$$

focusing on solving initial value problems (IVPs) with the following methods:
+ Forward Euler
+ Runge-Kutta 
+ Semi-implicit Euler Cromer

## Conservation Properties of ODE Solvers
We know that numerical methods will accrue numerical errors. While roundoff error will always occur, we have seen that the design of our numerical scheme can be used to minimize the truncation error. When we're solving physics problems, we often additionally require adherence to basic principles of conservation: laws of conservation of *mass, momentum, energy*. 

However, the ODE solver by itself does not know the context of the equation being solved, and thus error accumulation can in turn violate the governing physical principles of the problem we are trying to solve. 

Consider the second-order ODE system for an orbital trajectory from last week:

```{note} Ex: 2D Orbit ODEs
Recall from last week our system of ODEs for an orbit of a secondary object mass $m$ about a primary of mass $M$:

$$ \dot{\bf r}(t) = {\bf v}(t) = (u(t),v(t))$$

$$ \dot{\bf v}(t) = -\frac{GM}{r^3} {\bf r} $$

That we wish to solve for their trajectories:

$${\bf r}(t) = (x(t),y(t))$$ 
``` 

We can write the total specific angular momentum (we can use the specific or *per mass* values here as the secondary mass $m$ does not change) for our secondary as:
$$ j = |{\bf v} \times {\bf r} | = uy - vx $$
and the total specific energy as:
$$ e = \frac{1}{2} (u^2 + v^2) - \frac{GM}{x^2 + y^2}$$

If we take the forward Euler scheme:
$$ x_{i+1} = x_i + v_i \Delta t \\ v_{i+1} = v_i + \dot{v}_i \Delta t$$

And evaluate the angular momentum between timesteps:
$$j_{i+1} = u_{i+1} y_{i+1} - v_{i+1} x_{i+1} = (u_i + \dot{u}_i \Delta t )(y_i + v_{i} \Delta t) - (v_i + \dot{v}_i \Delta t )(x_i + u_{i} \Delta t) \\
j_{i+1} = (u_i y_i - v_i x_i) + \Delta t (\dot{u}_i y_i - \dot{v}_i x_i) + \Delta t^2 (\dot{u}_i v_i - \dot{v_i}u_i) \\
j_{i+1} = j_i +  \Delta t^2 (\dot{u}_i v_i - \dot{v_i}u_i)$$

using $\dot{u}_i y_i - \dot{v}_i x_i = \dot{\bf a} \times r = 0 $ for a central potential. We find that angular momentum is not conserved between timesteps!

### Symplectic Integration
How do we solve a problem that knows about conservation laws? *Symplectic* integrators are written to solve equations of motions in terms of conserved quantities using the Hamiltonian $H = T(p) + V(q)$ and the canonical coordinates for position and momentum $(q,p)$.

```{note} Hamiltonian Walkthrough
:class: dropdown
Recall that: 

$$ \dot{p} = -\frac{\partial H}{\partial q}; \ \ \ \dot{q} = \frac{\partial H}{\partial p} $$
thus:
$$ \dot{p} = -\frac{\partial V(q)}{\partial q}; \ \ \ \dot{q} = \frac{\partial T(p)}{\partial p}$$

For our orbital mechanics problem:

$$ H(p,q) = \frac{p^2}{2m} - GM/q $$

$$\dot{p} = -\frac{GM}{q^2} ; \ \ \dot{q} = \frac{p}{m}$$

which satisfies a time-independent Hamiltonian. 

Thus for small perturbations in $p$ and $q$ we can linearise (keeping perturbations to first order) to get:
$$p_{i+1} = p_i + \dot{p}_i \Delta t = p -\frac{GM}{(q_i + \dot{q}_i)^2} \Delta t = p_i -\frac{GM}{q_i}^2 \Delta t$$

$$q_{i+1} = q_i + \dot{q}_i \Delta t = q_i + \frac{p_i+\dot{p}_i}{m} \Delta t = q_i + \frac{p_{i+1}}{m}$$

Higher order schemes can be derived by splitting into multiple sub-steps. 
```
From the end result of the Hamiltonian derivation above, it turns out the **Euler-Cromer** method is actually a type of symplectic integrator. 

$$ v_{i+1} = v_i + \dot{v}_i \Delta t \ \ \ \ x_{i+1} = x_i + v_{i+1} \Delta t $$

We can see this play out by computing the angular momentum between timesteps:
$$j_{i+1} = u_{i+1} y_{i+1} - v_{i+1} x_{i+1} = u_{i+1}(y_i + v_{i+1} \Delta t) - v_{i+1} (x_i + u_{i+1} \Delta t) \\
 j_{i+1} = u_{i+1} y_i - v_{i+1} x_i = (u_i + \dot{u}_i \Delta t )y_i - (v_i + \dot{v}_i \Delta t ) x_i \\
 j_{i+1} = (u_i y_i - v_y x_i) + \Delta t( \dot{u}_i y_i - \dot{v}_i x_i) = j_i $$

To see that here we can conserve angular momentum to the same level of truncation error as the overall scheme. 

<h2><span class="fa fa-flash"></span> In-Class Coding Exercise </h2>

```{embed} icc_7-1
```
 
### Time-Symmetric Integration
An integrator is time-reversible or time-symmetric if you can iterate with the same scheme in reverse in order to obtain the same results. Time-symmetric integrators are great for problems with inherent periodicity, as they are energy conserving over full periodic cycles (but not between individual timesteps). 

Consider the Euler method:

$$ x_{i+1} = x_i + v_i \Delta t \\ v_{i+1} = v_i + \dot{v}_i \Delta t$$

Write in terms of a reverse step: take $\Delta t \rightarrow - \Delta t$ and $s_{i+1} \rightarrow s_{i-1}$

$$ x_{i-1} = x_i - v_i \Delta t \\ v_{i-1} = v_i - \dot{v}_i \Delta t$$

Reverse the method: re-arrange for $s_i$ in terms of $s_{i-1}$:

$$ x_{i} = x_{i-1} + v_i \Delta t \\ v_{i} = v_{i-1} + \dot{v}_i \Delta t$$

You can see that if we were to integrate forward, we would now end up with an implicit method to solve for the next timestep. 

### Leapfrog Method
Leapfrog is a second order time-symmetric method for ODES of the form:
$$dx/dt = f(x,t)$$
that uses information about $f(x,t)$ from an intermediate sub-step, similar to Runge-Kutta, but the sub-step update of $f(x,t)$ update is staggered from the computation of $x(t)$:

* For the initial step only: take an Euler step of $h/2$ to $x_{i+ 1/2}$

1. Take a step of width $h$ from $x_{i+1/2}$ to $x_{i+3/2}$ using $f_{i}$: 
$$x(t+\tfrac{3}{2}h)=x(t+\tfrac{1}{2}h)+h f(x(t+h),t+h)$$
2. Take a step of width $h$ from $x_{i+1}$ to $x_{i+2}$ using $f_{i+3/2}$:
$$x(t+2h)=x(t+h)+h f(x(t+\tfrac{3}{2}h),t+\tfrac{3}{2}h)$$

### Velocity Verlet Method
When used for 2nd order ODEs for kinematics, like in our orbital trajectory problem, we can write leap-frog in convenient  *kick-drift-kick* form:

$$v_{i+1/2} = v_i + \tfrac{1}{2} \dot{v}_i \Delta t, \\ 
x_{i+1} = x_i + v_{i+1/2} \Delta t, \\ 
v_{i+1} = v_{i+1/2} + \tfrac{1}{2} \dot{v}_{i+1} \Delta t$$

In which it is equivalent to velocity-verlet. 

Verlet is considered a **second order symplectic integrator**. 




<h2><span class="fa fa-flash"></span> In-Class Coding Exercise </h2>

```{embed} icc_7-2
```

## Additional Methods
Solving numerical ODEs is a well-established area of study. `scipy.integrate` is a library of implementations of various ODE solvers, which can be used on initial value problems with [`scipy.integrate.solve_ivp`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html) by selecting and ODE solver method. Many of these methods are Python wrappers for the famous Fortran [ODEPACK library](https://ui.adsabs.harvard.edu/abs/2019ascl.soft05021H/abstract) of widely used solvers in numerical works. These integrated library functions are a bit of a "black-box" -- they are written assuming that the user understands how they work and how to select appropriate methods and apply these functions. 

### Higher Order Methods
There are further variations on leapfrog that allow for higher order corrections, see Newman Ch. 8.5 for details
+ *Modified Midpoint Method*: formulates leapfrog without the starting half-step, making it easy to use adaptive time-stepping methods.
+ *Burlisch-Stoer Method*: adaptive timestepping of modified midpoint with Richardson Extrapolation (similar to [Romberg Integration](./4a_integrals.ipynb)). 

```{warning} 
Newman describes Burlisch-Stoer as a gold standard method, however there is a big caveat on how well-behaved your function has to be in order for Burlisch-Stoer to work. This caveat means, in practice, it is quite rare to have an ODE system for which Burlisch-Stoer is guaranteed to be the best option. This is why most library implementations will default offer a higher order Runge-Kutta scheme as the base option.
```

### Implicit Methods
Implicit methods discretize across time such that we must solve for the root of the equation for the next iteration. 
The simplest analogue to the explicit **Forward Euler** scheme: $ x_{i+1} = x_i + \Delta t \dot{x}_i $  
is the implicit **Backward Euler** scheme:
$$ x_{i+1} = x_i + \Delta t \ \dot{x}_{i+1}$$

(which we've already seen applied to just the position step in the aptly named semi-implicit Euler-Cromer scheme).

One of the most common use cases for numerical ODE solvers is for larger systems of coupled ODEs.

For example, consider this system for a reaction network of two species with concentrations $X_A$ and $X_B$ that are produced at rates $\alpha$ and $\beta$, respectively:

$$\frac{dX_A}{dt} = -\alpha X_A + \beta X_B,$$

$$\frac{dX_B}{dt} = \alpha X_A - \beta X_B$$

which can be written in matrix form:

$$\frac{d}{dt} \left ( \begin{array}{c}  X_A \\ X_B \end{array} \right )
             = \left ( \begin{array}{rr} -\alpha & \beta \\ \alpha  & -\beta  \end{array}   \right ) 
               \left ( \begin{array}{c} X_A \\ X_B \end{array} \right )$$

$$\dot{\bf X} = {\bf A}{\bf X}$$

in terms of an implicit  *backward Euler* scheme, we can write the implementation as:

$${\bf X}_{i+1} = {\bf X}_i + \Delta t {\bf A}{\bf X}_{i+1}$$

Which lets us write this in a familiar form (${\bf A}{\bf x} = {\bf v}$) as a [linear system of equations](./5a_lineareqs.ipynb):

$$(I - \Delta t {\bf A}){\bf X}_{i+1} = {\bf X}_i $$

for which we can use the methods discussed in week 5 to solve for the next timestep ${\bf X}_{i+1}$. 

Implicit methods are good for large ODE systems, specialized methods exist for *stiff* systems (in which the ODEs have vastly different rates), and for linearization of non-linear ODE systems.

:question
```{embed} ex_7-1
```